# Fire Detection Model Training — Smart Crowd Safety

**Trains a specialized YOLOv8 fire detection model on a real fire dataset using FREE GPU.**

## Before you start
1. Click **Runtime → Change runtime type → GPU** (T4 GPU is fine)
2. Click **Runtime → Run all**
3. Wait ~15-25 minutes
4. Download `fire_model.pt` at the end
5. Drop it into `backend/fire_model.pt` in your project
6. Restart backend

Expected output: **80-95% mAP** fire detection model.

## 1. Setup — verify GPU

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Download Fire Dataset

Uses a public fire detection dataset in YOLOv8 format (no API key needed).
If you want a larger dataset, swap the URL below for D-Fire or another.

In [ ]:
# Download a curated fire + smoke dataset (YOLOv8 format)
# This one has ~2500 labeled fire images
!mkdir -p /content/fire_dataset
%cd /content/fire_dataset

# Option A: Roboflow public fire dataset (no auth needed for public datasets)
!curl -L 'https://universe.roboflow.com/ds/fire-wrpgm/dataset/6/download/yolov8' -o fire.zip 2>/dev/null || echo 'Try alternative'

# Fallback: clone D-Fire from GitHub (large, ~21k images)
import os
if not os.path.exists('fire.zip') or os.path.getsize('fire.zip') < 1000:
    print('Roboflow URL failed. Using D-Fire dataset from GitHub instead...')
    !git clone https://github.com/gaiasd/DFireDataset.git
    print('D-Fire downloaded. You may need to convert format.')
else:
    !unzip -q fire.zip -d fire_data
    print('Dataset extracted.')
    !ls fire_data

In [ ]:
# Alternative: use kagglehub to download from Kaggle (requires signing in once)
# Uncomment if above doesn't work:
# !pip install -q kagglehub
# import kagglehub
# path = kagglehub.dataset_download('atulyakumar98/test-dataset')
# print('Path:', path)

## 3. Create data.yaml (if needed)

YOLOv8 needs a `data.yaml` pointing to train/val folders with `images/` and `labels/`.

In [ ]:
# Check if data.yaml already exists
import os
import glob

yaml_files = glob.glob('/content/fire_dataset/**/data.yaml', recursive=True)
print('Found data.yaml files:', yaml_files)

if yaml_files:
    DATA_YAML = yaml_files[0]
    print(f'Using: {DATA_YAML}')
    !cat {DATA_YAML}
else:
    # Create one manually (adjust paths based on what was downloaded)
    DATA_YAML = '/content/fire_dataset/data.yaml'
    yaml_content = '''path: /content/fire_dataset/fire_data
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['fire']
'''
    with open(DATA_YAML, 'w') as f:
        f.write(yaml_content)
    print(f'Created {DATA_YAML}')
    print(yaml_content)

## 4. Train YOLOv8 on GPU

- **Model**: YOLOv8s (good balance of speed and accuracy)
- **Epochs**: 100 with early stopping
- **Image size**: 640
- **Batch**: 16 (GPU can handle it)
- **Estimated time**: 15-25 minutes on T4 GPU

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8s as base
model = YOLO('yolov8s.pt')

# Train on fire dataset
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,              # GPU
    project='runs',
    name='fire_detector',
    exist_ok=True,
    patience=20,           # stop if no improvement for 20 epochs
    pretrained=True,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
)

## 5. View Training Results

In [ ]:
# Show metrics
from IPython.display import Image, display

results_img = '/content/runs/fire_detector/results.png'
if os.path.exists(results_img):
    display(Image(results_img, width=900))

confusion = '/content/runs/fire_detector/confusion_matrix.png'
if os.path.exists(confusion):
    display(Image(confusion, width=600))

## 6. Validate Model Performance

In [ ]:
# Load the best weights
best_model = YOLO('/content/runs/fire_detector/weights/best.pt')

# Run validation
metrics = best_model.val()
print(f'mAP50:     {metrics.box.map50:.2%}')
print(f'mAP50-95:  {metrics.box.map:.2%}')
print(f'Precision: {metrics.box.mp:.2%}')
print(f'Recall:    {metrics.box.mr:.2%}')

## 7. Download Trained Model

After running this cell, a download dialog opens. Save the file and:
1. Rename it to `fire_model.pt` (if not already)
2. Drop it into your project: `backend/fire_model.pt`
3. Restart the backend server

In [ ]:
from google.colab import files
import shutil

# Copy best weights to current directory with a clean name
src = '/content/runs/fire_detector/weights/best.pt'
dst = '/content/fire_model.pt'
shutil.copy(src, dst)

print(f'Model size: {os.path.getsize(dst) / 1024 / 1024:.1f} MB')
files.download(dst)

## 8. Test on a Sample Image (Optional)

Upload a fire image to test the model before downloading.

In [ ]:
from google.colab import files
import os

# Upload a test image
print('Upload a fire image to test...')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'\nTesting on {filename}')
    results = best_model(filename, conf=0.25)
    for r in results:
        print(f'Detections: {len(r.boxes)}')
        for box in r.boxes:
            conf = float(box.conf[0])
            print(f'  Fire at {conf:.2%} confidence')
        # Save annotated image
        r.save('output.jpg')
        display(Image('output.jpg', width=600))